In [1]:
!pip install google-generativeai


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import google.generativeai as genai

c:\Users\Dikshya\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Dikshya\AppData\Local\Temp\ipykernel_30128\613638648.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [3]:
from dotenv import load_dotenv
import os
import google.generativeai as genai

# Load .env file from Backend folder
load_dotenv("../.env")

API_KEY = os.getenv("GEMINI_API_KEY")

genai.configure(api_key=API_KEY)

model = genai.GenerativeModel("gemini-2.5-flash")

response = model.generate_content("Say Hello!")

print(response.text)

Hello!


In [4]:
import pandas as pd

df = pd.read_csv("../data/customer_support_tickets_clean.csv")
df.head()


,subject,body,answer,type,queue,priority,ticket_text,clean_text
0,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,"Account Disruption Dear Customer Support Team,...",account disruption dear customer support team ...
1,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,Query About Smart Home System Integration Feat...,query about smart home system integration feat...
2,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,Inquiry Regarding Invoice Details Dear Custome...,inquiry regarding invoice details dear custome...
3,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,Question About Marketing Agency Software Compa...,question about marketing agency software compa...
4,Feature Query,"Dear Customer Support,\n\nI hope this message ...",Thank you for your inquiry. Please specify whi...,Request,Technical Support,high,"Feature Query Dear Customer Support,\n\nI hope...",feature query dear customer support i hope thi...


In [5]:
sample_ticket = df.iloc[0]["ticket_text"]

print(sample_ticket)

Account Disruption Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.\n\nCould you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?


In [6]:
prompt = f"""
You are a professional Customer Support Agent.

Write a professional email reply.

Rules:

- Start with "Dear Customer,"
- Thank the customer for contacting support.
- Apologize for the inconvenience.
- Acknowledge the issue.
- Reassure the customer that the issue is being investigated.
- Do not invent information.
- Do not assume the cause.
- Keep the reply professional.
- End with:

Kind regards,
Customer Support Team

Customer Ticket:

{sample_ticket}
"""

response = model.generate_content(prompt)

print(response.text)

Dear Customer,

Thank you for contacting our support team.

We sincerely apologize for the significant inconvenience you are experiencing with the centralized account management portal being offline, which is currently blocking access to your account settings. We understand that this is causing substantial disruption.

Please be assured that our team is actively investigating this issue as a matter of urgent priority. We are working diligently to identify the root cause and restore full functionality as quickly as possible.

We are currently unable to provide an estimated time for resolution or alternative ways to access and manage your account during this downtime. We will provide an update as soon as more information becomes available regarding the status, an estimated time for resolution, and any potential alternative solutions.

Thank you for your patience and understanding as we work to resolve this.

Kind regards,
Customer Support Team


In [7]:
sample_df = df.sample(5, random_state=42)

for i, row in sample_df.iterrows():

    sample_ticket = row["ticket_text"]

    prompt = f"""
You are a professional Customer Support Agent.

Write a polite, professional, and empathetic email reply to the customer.

Guidelines:
- Start with "Dear Customer,"
- Thank the customer for contacting support.
- Apologize for the inconvenience.
- Acknowledge the customer's issue.
- Reassure the customer that the support team is actively investigating the issue.
- If the customer requested an update, mention that updates will be shared as soon as they become available.
- Do NOT invent technical details.
- Do NOT assume the root cause of the issue.
- Do NOT promise an estimated resolution time unless it is explicitly provided.
- Do NOT ask unnecessary questions.
- Keep the reply concise (5–7 sentences).
- End exactly with:

Kind regards,
Customer Support Team

Customer Ticket:

{sample_ticket}
"""

    response = model.generate_content(prompt)

    print("=" * 100)
    print("ORIGINAL TICKET:\n")
    print(sample_ticket[:400], "...\n")

    print("GENERATED REPLY:\n")
    print(response.text)

ORIGINAL TICKET:

Request for Documentation on Integrating Adobe Sign Looking for detailed instructions on integrating the Adobe Sign project management SaaS solution. Could you outline the steps and requirements needed for a successful integration? ...

GENERATED REPLY:

Dear Customer,

Thank you for contacting our support team. We apologize for any inconvenience you may have experienced in locating the detailed instructions for integrating Adobe Sign. We understand you are looking for comprehensive documentation outlining the steps and requirements for a successful integration. Please be assured that our team is actively investigating and gathering the necessary information regarding Adobe Sign integration. We will share the requested documentation and steps with you as soon as they become available.

Kind regards,
Customer Support Team
ORIGINAL TICKET:

Support Concerning Security Incident An unauthorized access attempt was detected in the healthcare data system. This may have occur

In [8]:
# Select a sample ticket

sample_row = df.sample(1, random_state=42).iloc[0]

ticket_type = sample_row["type"]
queue = sample_row["queue"]
priority = sample_row["priority"]
sample_ticket = sample_row["ticket_text"]


# Prompt

prompt = f"""
You are an AI Customer Support Copilot assisting human support agents.

Your task is to generate a professional draft reply that a human support agent can review before sending.

Customer Ticket Type:
{ticket_type}

Assigned Support Queue:
{queue}

Priority:
{priority}

Customer Ticket:
{sample_ticket}

Instructions:

- Start with "Dear Customer,"
- Thank the customer for contacting support.
- If the ticket describes a problem or incident, apologize for the inconvenience.
- If the ticket is a request or question, do NOT apologize unnecessarily.
- Acknowledge the customer's concern or request.
- Adapt the response based on the ticket type:
    • Request → acknowledge the request professionally.
    • Question → provide a helpful response if possible.
    • Problem → reassure the customer that the appropriate support team is reviewing the issue.
    • Incident → acknowledge the urgency and reassure the customer.
- Maintain a polite, empathetic, and professional tone.
- Never invent technical details.
- Never assume the root cause of an issue.
- Never promise an estimated resolution time unless explicitly provided.
- Never ask unnecessary questions.
- Keep the reply between 5 and 7 sentences.
- End exactly with:

Kind regards,
Customer Support Team

Return ONLY the email reply.
"""


# Generate Reply

response = model.generate_content(prompt)

print("=" * 100)
print("TICKET TYPE:", ticket_type)
print("QUEUE:", queue)
print("PRIORITY:", priority)
print("=" * 100)
print("ORIGINAL TICKET:\n")
print(sample_ticket)
print("\n" + "=" * 100)
print("GENERATED REPLY:\n")
print(response.text)

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 40.040271876s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 5
}
, retry_delay {
  seconds: 40
}
]